### ***CNN***
- ***CNN is a neural network designed to learn spatial patterns using convolutional filters. It is widely used for image and computer vision tasks.***

- ***CNN is preferred for images because it preserves spatial relationships and learns local patterns such as edges, textures, and shapes.***

- ***It uses local connectivity and weight sharing, which significantly reduces the number of parameters compared with a fully connected network.***

- ***As the network becomes deeper, these low-level features can be combined into higher-level features.***

In [1]:
import torch
import torchvision

import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import CIFAR10

In [2]:
from torch._C import TracingState
## Dataset & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms # resize img

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

In [3]:
# DataLoaders
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size=64)

In [7]:
## Build Our CNN Model
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()

    self.cov_layers = nn.Sequential(
        # 1st layer
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        # 2nd layer
        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        # 3rd layer
        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2,2),
    )

    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128,256),
        nn.ReLU(),
        nn.Linear(256,10)
    )
  # forward propagation
  def forward(self,x):
    x = self.cov_layers(x)
    x = x.view(x.size(0),-1) # flatten
    x = self.fc_layers(x)
    return x

In [8]:
model = CNN()

criterian = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [9]:
## Train Our CNN
epochs = 10
for epoch in range(epochs):
  train_loss = 0.0
  for images,labels in trainloader:
    optimizer.zero_grad()

    # forward propagation
    output = model.forward(images)
    loss = criterian(output,labels)

    # backward propagation
    loss.backward()
    optimizer.step()
    train_loss+=loss.item()
print(f"epoch:{epoch+1}/{epochs} & loss:{train_loss/len(trainloader)}")


epoch:10/10 & loss:0.20894629218141594


In [11]:
from functools import total_ordering
## Evaluate Our CNN
correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
  for images,labels in testloader:
    outputs = model.forward(images)

    _,predicted = torch.max(outputs,1)
    correct_labels+=(predicted==labels).sum().item()
    total_labels+=labels.size(0)

print(f"accuracy:{correct_labels/total_labels*100}")

accuracy:77.35


In [15]:
# save our model
saved_model = "cnn_model.pt"
torch.save(model.state_dict(),saved_model)
print(f"model saved success ✅ {saved_model}")

model saved success ✅ cnn_model.pt


### ***FNN***
- ***FNN is the simplest neural-network architecture where information moves only forward.***

In [12]:
import torch
import torch.nn as nn

class FNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),

            nn.Linear(16, 8),
            nn.ReLU(),

            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.network(x)

In [13]:
model = FNN()
print(model)

FNN(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
  )
)


In [14]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# for epoch in range(100):
#     optimizer.zero_grad()

#     output = model(X_train)
#     loss = criterion(
#         output,
#         y_train
#     )

#     loss.backward()
#     optimizer.step()